#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

#1. Read from bronze layer

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

#2. Silver Transformation

##2.1 Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##2.2 Customer ID Cleanup

In [0]:
df = df.withColumn("CID", 
                   F.when(col("CID").startswith("NAS"), F.substring(col("CID"), 4, F.length(col("CID"))))
                   .otherwise(col("CID"))                                  
                   )
                   

##2.3 Birthdate Validation

In [0]:

df = df.withColumn("BDATE", 
                   F.when(col("BDATE") > F.current_date() , None)
                    .otherwise(col("BDATE"))
                   )

##2.4 Gender Normalization

In [0]:
df = df.withColumn(
    "GEN"
    , F.when(col("GEN").isin("M", "Male"), "Male")
        .when(col("GEN").isin("F", "Female"), "Female")
        .otherwise("n/a")

)

##2.3 Rename Columns

In [0]:
Renamed_Map = {
    "CID": "customer_nember",
    "BDATE": "Birth_date",
    "GEN": "gender"
}
for old_name, new_name in Renamed_Map.items():
    df = df.withColumnRenamed(old_name, new_name)


##2.4 dataframe sanity check

In [0]:
df.limit(10).display()

#3. Write to silver layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

##3.1 sanity check of the silver table

In [0]:
%sql
select * from workspace.silver.erp_customers limit 10